# Recover, Retarget, Guide — the whole pipeline on your own clip

From one reference video and one caption to a generated video, with no human step: Module 1 (Recover), Module 2 (Retarget, automatic path) and Module 3 (Guide), as the thesis's test set was run.

**Three runtimes.** The three modules need three incompatible environments (the tracker, the image generator with the 3D reconstructors, the video generator), so this notebook is in three parts and you **restart the runtime between them** (Runtime → Disconnect and delete runtime, then reopen and run the settings cell again). Everything is kept on your Google Drive, so nothing is lost between parts.

**What you need before starting**
- A Colab runtime with a GPU: 24 GB (L4) is enough for parts 1 and 2; part 3 needs an A100 (40 GB).
- **Your own Hugging Face account**, because three of the models are gated: open the pages of `black-forest-labs/FLUX.1-Depth-dev` and of `facebook/sam-3d-body-dinov3` (human targets) or `facebook/sam-3d-objects` (vehicle targets), accept their terms while signed in, create an access token in your account settings, and store it as the Colab secret `HF_TOKEN` (the key icon on the left). The download of a gated model only succeeds for an account that accepted the terms, and the token is how the runtime proves it is yours. Everything else is public and needs no token.
- Your clip as an mp4 (at least 24 frames; the pipeline reads up to 81) and its target category: `human` (the target is a man in a grey jacket and dark trousers, walking) or `vehicle` (a red car, driving). These two are the category proxies the thesis validated; the proxy fixes the target's identity, so the subject words of your caption do not change it.
- A caption of your clip, one sentence, in this template: **`<subject> <motion verb> <preposition> <scene>`**, for example `a camel walking through a sandy enclosure with wooden fences` or `a cyclist rides along a coastal road at sunset`. The rule reads the first motion verb (walks, runs, drives, rides, travels, gallops, traverses, moves, swims, skates, jumps, flies, sails, climbs, crosses, and their -s forms), then the scene from the first preposition after it (through, along, across, down, past, around, into, over, on, in, at, near, between, ...) up to the first full stop or a clause opener such as `, with`. That scene becomes the target scene in the prompt; the rule is applied as written and never edited by hand.

**Time.** Part 1 about 20 minutes; part 2 about 30 minutes for a human target, several hours the first time for a vehicle target (the SAM 3D Objects environment is built from source and then saved to Drive); part 3 about 20 minutes plus the first download of Wan 2.1.

A clip can be refused by a check at any stage (camera, subject, support surface); that is a recorded outcome, not an error, and the status cell says where.

In [ ]:
#@title S · Settings (run first in every part)
REPO_URL = "https://github.com/jnheinrich451-eng/Recover-Retarget-Guide.git"  #@param {type:"string"}
PROXY_ASSET_URL = "https://github.com/jnheinrich451-eng/Recover-Retarget-Guide/releases/download/v1.0/proxies_of_record.zip"  #@param {type:"string"}
CLIP_ID = "my_clip"  #@param {type:"string"}
CATEGORY = "human"  #@param {type:"string"}
# CATEGORY: "human" or "vehicle" (the thesis's two targets), or a new name such as "bear" together with the two fields below
NEW_SUBJECT_OBJECT = ""  #@param {type:"string"}
NEW_SUBJECT_KIND = "rigid"  #@param ["rigid", "human"]
# NEW_SUBJECT_OBJECT: the new subject's identity, e.g. "a brown bear"; NEW_SUBJECT_KIND: rigid (SAM 3D Objects, rendered along
#   its path) or human (SAM 3D Body, camera only). Experimental: outside the thesis's evaluated pipeline.
CAPTION = "a camel walking through a sandy enclosure with wooden fences"  #@param {type:"string"}
DATA_ROOT = "/content/drive/MyDrive/rrg_data"  #@param {type:"string"}
SOURCE_CLIP = "/content/drive/MyDrive/my_clip.mp4"  #@param {type:"string"}
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DATA = Path(DATA_ROOT); (DATA / "videos").mkdir(parents=True, exist_ok=True)
os.environ["MT_ROOT"] = str(DATA); os.environ["MT_DATA"] = str(DATA); os.environ["SKIP_VIDEOS"] = "1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN: set from the Colab secret")
except Exception as e:
    print("HF_TOKEN: not set (Module 2's gated models will fail to download):", e)
REPO_DIR = Path("/content/repo")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
target = DATA / "videos" / f"{CLIP_ID}.mp4"
if not target.exists():
    assert Path(SOURCE_CLIP).exists(), f"put your clip at {SOURCE_CLIP} (or change SOURCE_CLIP)"
    shutil.copyfile(SOURCE_CLIP, target)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("data root:", DATA, "| clip:", target.name, "| category:", CATEGORY, "| gpu:", gpu or "none")
RUN = [sys.executable, "scripts/run_all.py", "--env", "colab_public", "--clip", CLIP_ID, "--category", CATEGORY]
def status():
    subprocess.run(RUN + ["--status"])

## One click

Run **S** and then the cell below, and wait. It builds the three environments side by side in this runtime, screens your clip (Module 1 and the checks the thesis's test set passed), downloads the proxies of record, makes a new subject's proxy if you named one, and runs Modules 2 and 3 on the automatic path. If the runtime disconnects, run S and the cell again: finished stages are skipped. The three-part path below does the same in three runtimes and is the fallback if an environment refuses to share a VM.


In [ ]:
#@title ONE CLICK · everything in one runtime (after S): environments, screening, proxy, Modules 2 and 3
# Three environments in one VM: two virtual environments on Colab's torch (Phase 1: the tracker; Phase 2: the generators and SAM 3D
# Body) and the SAM 3D Objects environment built by scripts/setup_sam3d.sh. Every step is resumable: run the cell again after a
# disconnect and finished stages are skipped. Expect 1-2 hours for a human target the first time (longer for a vehicle target, whose
# reconstructor environment is compiled once and then saved to Drive). Needs an A100 for Module 3.
import json, shlex
V1, V2 = Path("/content/venv-phase1"), Path("/content/venv-phase2")

def sh(cmd, **kw):
    extra = kw.pop("env", {})
    print("\n$", cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd), flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), env={**os.environ, **extra}, **kw)
    assert r.returncode == 0, "stopped here; read the log above"

cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "8.0"
# 1. the two environments
for v in (V1, V2):
    if not (v / "bin" / "python").exists():
        sh([sys.executable, "-m", "venv", "--system-site-packages", str(v)])
sh(f"source {V1}/bin/activate && SKIP_VIDEOS=1 bash scripts/bootstrap.sh", executable="/bin/bash")
sh(f"source {V2}/bin/activate && SKIP_VIDEOS=1 bash scripts/bootstrap_phase2.sh && (python -c 'import sam2' 2>/dev/null || pip install -q 'git+https://github.com/facebookresearch/sam2.git')", executable="/bin/bash")
KIND = {"human": "human", "vehicle": "rigid"}.get(CATEGORY, NEW_SUBJECT_KIND)
sh(f"source {V2}/bin/activate && bash scripts/setup_sam3d.sh {'body' if KIND == 'human' else 'both'}", executable="/bin/bash", env={"DRIVE_ROOT": str(DATA), "TORCH_CUDA_ARCH_LIST": cap})
P1, P2 = str(V1 / "bin" / "python"), str(V2 / "bin" / "python")
# 2. screening: Module 1 and the admission, source handoff, camera gate and support-surface checks (a FAIL is a recorded outcome)
sh([P1, "scripts/run_census.py", "--clips", CLIP_ID, "--category", CATEGORY, "--caption", CAPTION, "--drive-root", str(DATA), "--videos-dir", str(DATA / "videos")])
rows = json.loads((DATA / "reports" / "census" / "census.json").read_text(encoding="utf-8"))
verdict = next((r for r in rows if r["clip"] == CLIP_ID), {}).get("census", "?")
print("\ncensus verdict for", CLIP_ID, "->", verdict)
assert verdict == "PASS", "the clip did not pass the screening; the reasons are in reports/census/census.md under your data root"
# 3. the proxies of record, and a new subject's proxy if the category is new
if not (DATA / "cache" / "m2_b2").exists():
    import urllib.request, zipfile
    z = Path("/content/proxies.zip"); urllib.request.urlretrieve(PROXY_ASSET_URL, z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA)
if CATEGORY not in ("human", "vehicle"):
    assert NEW_SUBJECT_OBJECT, "a new subject needs NEW_SUBJECT_OBJECT in the settings cell"
    sh([P2, "scripts/make_proxy.py", "--category", CATEGORY, "--object", NEW_SUBJECT_OBJECT, "--kind", KIND, "--env", "colab_oneclick", "--drive-root", str(DATA)])
# 4. Modules 2 and 3 on the automatic path (the interpreters per step come from the colab_oneclick preset)
RUN1 = [P2, "scripts/run_all.py", "--env", "colab_oneclick", "--clip", CLIP_ID, "--category", CATEGORY]
sh(RUN1 + ["--module", "2", "--auto", "--proxy-from", str(DATA), "--caption", CAPTION])
sh(RUN1 + ["--module", "3", "--auto", "--m3-config", "configs/m3_batch_exp.yaml", "--seeds", "0"])
subprocess.run(RUN1 + ["--status"])
out_dir = DATA / "cache" / "m3" / CLIP_ID / CATEGORY
vids = sorted(out_dir.glob("A*_seed0_*.mp4")) if out_dir.exists() else []
print("\nvideos:", *[v.name for v in vids], sep="\n  ")
import base64
from IPython.display import HTML, display

def tag(p, label):
    b = base64.b64encode(Path(p).read_bytes()).decode()
    return f"<div style='display:inline-block;margin:6px;text-align:center'><video src='data:video/mp4;base64,{b}' width='416' autoplay loop muted playsinline></video><br>{label}</div>"

display(HTML("".join(tag(v, v.name) for v in vids)))


## Part 1 · Recover (Module 1): tracks, depth, camera, subject mask, heading
Environment: `scripts/bootstrap.sh` (OpenD4RT, ViPE, SAM 2.1). Output: `cache/stage_*/<clip>/` under your data root.

In [ ]:
#@title 1 · Module 1
assert subprocess.run(["bash", "scripts/bootstrap.sh"], env=os.environ).returncode == 0, "bootstrap failed; read the log above"
r = subprocess.run(RUN + ["--module", "1"], env=os.environ)
status()
print("\nPart 1 done" if r.returncode == 0 else "\nPart 1 stopped: read the log; a refusal names the check")
print("Next: Runtime > Disconnect and delete runtime; reopen the notebook; run S, then Part 2.")

## Part 2 · Retarget (Module 2, automatic path): proxy, placement, control depth, anchor, reconstruction, handoff
Environment: `scripts/bootstrap_phase2.sh` (FLUX.1 Depth through diffusers) and `scripts/setup_sam3d.sh` (SAM 3D Body for human targets; SAM 3D Objects for vehicle targets, built once and saved to Drive). The category proxies of record are downloaded from the release asset and copied, never regenerated, as in the thesis's experiment. For a new subject, `scripts/make_proxy.py` first makes its proxy: candidate images from the fixed template, a mask, a reconstruction by kind, and the registration in `configs/subjects.yaml`.


In [ ]:
#@title 2 · Module 2
import urllib.request, zipfile
if not (DATA / "cache" / "m2_b2").exists():
    z = Path("/content/proxies.zip"); urllib.request.urlretrieve(PROXY_ASSET_URL, z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA)
    print("proxies of record unpacked under", DATA)
assert subprocess.run(["bash", "scripts/bootstrap_phase2.sh"], env=os.environ).returncode == 0, "bootstrap failed; read the log above"
os.environ["DRIVE_ROOT"] = str(DATA)       # setup_sam3d.sh saves the built environment under <DRIVE_ROOT>/envs for later sessions
KIND = {"human": "human", "vehicle": "rigid"}.get(CATEGORY, NEW_SUBJECT_KIND)
assert subprocess.run(["bash", "scripts/setup_sam3d.sh", "body" if KIND == "human" else "both"], env=os.environ).returncode == 0, "SAM 3D set-up failed; read the log above"
if CATEGORY not in ("human", "vehicle"):
    assert NEW_SUBJECT_OBJECT, "a new subject needs NEW_SUBJECT_OBJECT in the settings cell"
    r = subprocess.run([sys.executable, "scripts/make_proxy.py", "--category", CATEGORY, "--object", NEW_SUBJECT_OBJECT, "--kind", KIND, "--drive-root", str(DATA)], env=os.environ)
    assert r.returncode == 0, "the proxy was not made; read the log above (the overlay under probe/m2d/proxies shows the candidate and its mask)"
r = subprocess.run(RUN + ["--module", "2", "--auto", "--proxy-from", str(DATA), "--caption", CAPTION], env=os.environ)
status()
print("\nPart 2 done" if r.returncode == 0 else "\nPart 2 stopped: read the log; a refusal names the gate")
print("Next: Runtime > Disconnect and delete runtime; reopen on an A100; run S, then Part 3.")

## Part 3 · Guide (Module 3): lift, camera, renders, guided sampling, grading
Environment: `scripts/bootstrap_phase2.sh`. Runtime: A100 (40 GB). Output: the backbone's video and ours under `cache/m3/<clip>/<category>/`, the grader's table under `reports/m3_s4/`.

In [ ]:
#@title 3 · Module 3
assert subprocess.run(["bash", "scripts/bootstrap_phase2.sh"], env=os.environ).returncode == 0, "bootstrap failed; read the log above"
r = subprocess.run(RUN + ["--module", "3", "--auto"], env=os.environ)
status()
out_dir = DATA / "cache" / "m3" / CLIP_ID / CATEGORY
vids = sorted(out_dir.glob("A*_seed0_*.mp4")) if out_dir.exists() else []
print("\nvideos:", *[v.name for v in vids], sep="\n  ")
import base64
from IPython.display import HTML, display
def tag(p, label):
    b = base64.b64encode(Path(p).read_bytes()).decode()
    return f"<div style='display:inline-block;margin:6px;text-align:center'><video src='data:video/mp4;base64,{b}' width='416' autoplay loop muted playsinline></video><br>{label}</div>"
display(HTML("".join(tag(v, v.name) for v in vids)))